# 🧠 Semantic segmentation and 3D U-Net

**Xu Lab | Biology-to-CryoEM Research | Module 14**  
**Phase:** D: Learning from tomograms | **Level:** scaffolded undergraduate → research | **Time:** 60–100 minutes  
**Hardware:** Google Colab free-tier CPU (optional T4 GPU for later extensions)  
**Core tools:** Python, NumPy, SciPy, Matplotlib; selected modules use scikit-learn/PyTorch  
**Prerequisites:** Previous tutorial  
**Status:** self-contained *pedagogical demonstration* using synthetic data, not an official reproduction of Xu Lab publications.  
**Safety:** for research and educational purposes only, not for clinical diagnosis.

## Learning objectives

1. Explain the biological question and physical limitations.
2. Derive or interpret the model below and identify its assumptions.
3. Run the experiment, change a parameter, and interpret quantitative output.
4. Explain how the concept supports the research papers linked below.

## Biological motivation

Classifying an entire particle differs from assigning a semantic class to every voxel.

## Conceptual bridge

Create dense labels, compute Dice/IoU, and compare a 3D convolution block with a voxel-wise threshold baseline.

## Mathematical model

$\operatorname{Dice}(P,G)=2|P\cap G|/(|P|+|G|)$.

## Notebook roadmap

1. Reproducible setup  2. Synthetic experiment  3. Interpretation  4. Challenge  5. Paper connection.


In [ ]:
# -- Environment check and shared utilities --
import sys, importlib.util
import numpy as np
import scipy
from scipy import ndimage as ndi, signal
import matplotlib.pyplot as plt
SEED=2026
rng=np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi':95,'axes.grid':False})
print('Python:',sys.version.split()[0], 'NumPy:',np.__version__)
print('PyTorch available:',bool(importlib.util.find_spec('torch')))
try:
    import torch
    print('CUDA available:',torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch not installed; required only for Module 13/14')
# Colab stock environment normally contains the required libraries.
# If an import fails, run: %pip install -q numpy scipy matplotlib scikit-learn torch

def phantom(n=48):
    """Build a deterministic 3D synthetic density with three distinct macromolecular blobs."""
    zz, yy, xx = np.mgrid[:n,:n,:n].astype(float)
    rho = np.zeros((n,n,n),dtype=float)
    for z,y,x,s,amp in [(0.45*n,.47*n,.48*n,.10*n,1.),(.65*n,.38*n,.58*n,.07*n,.7),(.40*n,.66*n,.36*n,.075*n,.8)]:
        rho += amp*np.exp(-((zz-z)**2+(yy-y)**2+(xx-x)**2)/(2*s*s))
    return rho/np.max(rho)


## Experiment: build, inspect, and check a scientific claim

Run the code below, then identify (i) input, (ii) transformation, (iii) output and (iv) a key assumption.

In [ ]:
v=phantom(40);gt=v>.32
obs=v+rng.normal(0,.14,v.shape)
pred=obs>.3
inter=(pred&gt).sum();dice=2*inter/(pred.sum()+gt.sum());iou=inter/(pred|gt).sum()
print('Toy voxel Dice, IoU',round(dice,3),round(iou,3))
fig,ax=plt.subplots(1,3,figsize=(9,3))
for a,x,t in zip(ax,[obs[20],gt[20],pred[20]],['Noisy','True mask','Predicted mask']): a.imshow(x,cmap='gray');a.set_title(t);a.axis('off')
plt.show();assert 0<=dice<=1

## Beyond the threshold baseline: an actual tiny 3D U-Net

A **3D U-Net** learns spatially contextual voxel predictions using an encoder, downsampling, upsampling and skip connections. The following compact network uses a single training step to verify forward shape, backpropagation and Dice-style evaluation; the one-step output is **not** a trained research model. For a full optimization experiment, consult the linked existing `sci_img` segmentation tutorials.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(SEED)
torch.set_num_threads(2)
class TinyUNet3D(nn.Module):
    """Minimal volume-to-volume U-Net for dimension and gradient demonstrations."""
    def __init__(self):
        super().__init__()
        self.enc=nn.Sequential(nn.Conv3d(1,4,3,padding=1),nn.ReLU(),nn.Conv3d(4,4,3,padding=1),nn.ReLU())
        self.pool=nn.MaxPool3d(2)
        self.bottleneck=nn.Sequential(nn.Conv3d(4,8,3,padding=1),nn.ReLU())
        self.up=nn.Upsample(scale_factor=2,mode='trilinear',align_corners=False)
        self.dec=nn.Sequential(nn.Conv3d(12,4,3,padding=1),nn.ReLU(),nn.Conv3d(4,1,1))
    def forward(self,x):
        skip=self.enc(x)
        bottleneck=self.bottleneck(self.pool(skip))
        return self.dec(torch.cat([skip,self.up(bottleneck)],dim=1))
small_x=torch.tensor(obs[::2,::2,::2],dtype=torch.float32)[None,None]
small_y=torch.tensor(gt[::2,::2,::2],dtype=torch.float32)[None,None]
model=TinyUNet3D()
optimizer=torch.optim.Adam(model.parameters(),lr=.002)
logits=model(small_x)
loss=nn.BCEWithLogitsLoss()(logits,small_y)
optimizer.zero_grad();loss.backward();optimizer.step()
print('3D U-Net input/output:',tuple(small_x.shape),tuple(logits.shape),'loss:',round(loss.item(),3))
assert logits.shape==small_y.shape

## Explain what happened

**Expected scientific observation:** Overlap alone can hide disconnected membranes, merged instances or topology errors.

Write 3–5 sentences connecting the plotted result to the biological motivation. Describe one failure case and one alternative explanation.

## Guided challenge (edit code above)

Choose thresholds by validation rather than testing data; compare boundary quality when two objects touch.

**Deliverable:** a labeled figure or small results table, a claim, and one limitation. Quantify the effect rather than just saying “it looks better.”

## Research connection and reading strategy

**Related Xu Lab research:** 2018 supervised 3D segmentation; 2025 Vox-UDA; 2026 TopoFuse.

Read each paper in this order: biological question → observations/data → forward model → key objective → baseline → evaluation split → limitations. Distinguish what the toy exercise illustrates from the actual published implementation. See `../PAPER_TO_TUTORIAL_MAP.md` for publication titles and further prerequisites.

## Exit ticket

1. What is known from the measured data, versus assumed by the model?
2. Which plotted quantity could be misleading if evaluated only on synthetic data?
3. Sketch one controlled ablation and one biological validation experiment.
4. Save an updated notebook and a 150-word lab note.